In [ ]:
import os
import numpy as np
import pickle
import torch
from argparse import ArgumentParser
from tqdm import tqdm
import glob

from articulate.model import ParametricModel
from articulate import math
from config import paths, datasets


In [2]:
# specify target FPS
TARGET_FPS = 30

# left wrist, right wrist, left thigh, right thigh, head, pelvis
vi_mask = torch.tensor([1961, 5424, 876, 4362, 411, 3021])
ji_mask = torch.tensor([18, 19, 1, 2, 15, 0])
body_model = ParametricModel(paths.smpl_file)


In [ ]:


def _syn_acc(v, smooth_n=4):
    """Synthesize accelerations from vertex positions."""
    mid = smooth_n // 2
    scale_factor = TARGET_FPS ** 2 

    acc = torch.stack([(v[i] + v[i + 2] - 2 * v[i + 1]) * scale_factor for i in range(0, v.shape[0] - 2)])
    acc = torch.cat((torch.zeros_like(acc[:1]), acc, torch.zeros_like(acc[:1])))

    if mid != 0:
        acc[smooth_n:-smooth_n] = torch.stack(
            [(v[i] + v[i + smooth_n * 2] - 2 * v[i + smooth_n]) * scale_factor / smooth_n ** 2
             for i in range(0, v.shape[0] - smooth_n * 2)])
    return acc

def process_amass():
    def _foot_ground_probs(joint):
        """Compute foot-ground contact probabilities."""
        dist_lfeet = torch.norm(joint[1:, 10] - joint[:-1, 10], dim=1)
        dist_rfeet = torch.norm(joint[1:, 11] - joint[:-1, 11], dim=1)
        lfoot_contact = (dist_lfeet < 0.008).int()
        rfoot_contact = (dist_rfeet < 0.008).int()
        lfoot_contact = torch.cat((torch.zeros(1, dtype=torch.int), lfoot_contact))
        rfoot_contact = torch.cat((torch.zeros(1, dtype=torch.int), rfoot_contact))
        return torch.stack((lfoot_contact, rfoot_contact), dim=1)

    # enable skipping processed files
    try:
        processed = [fpath.name for fpath in (paths.processed_datasets).iterdir()]
    except FileNotFoundError:
        processed = []

    for ds_name in datasets.amass_datasets:
        # skip processed 
        if f"{ds_name}.pt" in processed:
            continue

        data_pose, data_trans, data_beta, length = [], [], [], []
        print("\rReading", ds_name)

        for npz_fname in tqdm(sorted(glob.glob(os.path.join(paths.raw_amass, ds_name, "*/*_poses.npz")))):
            try: cdata = np.load(npz_fname)
            except: continue

            framerate = int(cdata['mocap_framerate'])
            if framerate not in [120, 60, 59]:
                continue

            # enable downsampling
            step = max(1, round(framerate / TARGET_FPS))

            data_pose.extend(cdata['poses'][::step].astype(np.float32))
            data_trans.extend(cdata['trans'][::step].astype(np.float32))
            data_beta.append(cdata['betas'][:10])
            length.append(cdata['poses'][::step].shape[0])

        if len(data_pose) == 0:
            print(f"AMASS dataset, {ds_name} not supported")
            continue

        length = torch.tensor(length, dtype=torch.int)
        print(f"Total frames in {ds_name}: {length}")
        shape = torch.tensor(np.asarray(data_beta, np.float32))
        tran = torch.tensor(np.asarray(data_trans, np.float32))
        pose = torch.tensor(np.asarray(data_pose, np.float32)).view(-1, 52, 3)

        # include the left and right index fingers in the pose
        pose[:, 23] = pose[:, 37]     # right hand 
        pose = pose[:, :24].clone()   # only use body + right and left fingers

        # align AMASS global frame with DIP
        amass_rot = torch.tensor([[[1, 0, 0], [0, 0, 1], [0, -1, 0.]]])
        tran = amass_rot.matmul(tran.unsqueeze(-1)).view_as(tran)
        pose[:, 0] = math.rotation_matrix_to_axis_angle(
            amass_rot.matmul(math.axis_angle_to_rotation_matrix(pose[:, 0])))

        print("Synthesizing IMU accelerations and orientations")
        b = 0
        out_pose, out_shape, out_tran, out_joint, out_vrot, out_vacc, out_contact = [], [], [], [], [], [], []
        for i, l in tqdm(list(enumerate(length))):
            if l <= 12: b += l; print("\tdiscard one sequence with length", l); continue
            p = math.axis_angle_to_rotation_matrix(pose[b:b + l]).view(-1, 24, 3, 3)
            grot, joint, vert = body_model.forward_kinematics(p, shape[i], tran[b:b + l], calc_mesh=True)

            out_pose.append(p.clone())  # N, 24, 3, 3
            out_tran.append(tran[b:b + l].clone())  # N, 3
            out_shape.append(shape[i].clone())  # 10
            out_joint.append(joint[:, :24].contiguous().clone())  # N, 24, 3
            out_vacc.append(_syn_acc(vert[:, vi_mask]))  # N, 6, 3
            out_contact.append(_foot_ground_probs(joint).clone()) # N, 2

            out_vrot.append(grot[:, ji_mask])  # N, 6, 3, 3
            b += l

        print("Saving...")
        data = {
            'joint': out_joint,
            'pose': out_pose,
            'shape': out_shape,
            'tran': out_tran,
            'acc': out_vacc,
            'ori': out_vrot,
            'contact': out_contact
        }
        data_path = paths.processed_datasets / f"{ds_name}.pt"
        torch.save(data, data_path)
        print(f"Synthetic AMASS dataset is saved at: {data_path}")



In [ ]:

def process_totalcapture():
    """Preprocess TotalCapture dataset for testing."""

    inches_to_meters = 0.0254
    pos_file = 'gt_skel_gbl_pos.txt'
    ori_file = 'gt_skel_gbl_ori.txt'

    subjects = ['S1', 'S2', 'S3', 'S4', 'S5']

    # Load poses from processed AMASS dataset
    amass_tc = torch.load(os.path.join(paths.processed_datasets, "AMASS", "TotalCapture", "pose.pt"))
    tc_poses = {pose.shape[0]: pose for pose in amass_tc}

    processed, failed_to_process = [], []
    accs, oris, poses, trans = [], [], [], []
    for file in sorted(os.listdir(paths.calibrated_totalcapture)):
        if not file.endswith(".pkl") or ('s5' in file and 'acting3' in file) or not any(file.startswith(s.lower()) for s in subjects):
            continue

        data = pickle.load(open(os.path.join(paths.calibrated_totalcapture, file), 'rb'), encoding='latin1')
        ori = torch.from_numpy(data['ori']).float()
        acc = torch.from_numpy(data['acc']).float()

        # Load pose data from AMASS
        try: 
            name_split = file.split("_")
            subject, activity = name_split[0], name_split[1].split(".")[0]
            pose_npz = np.load(os.path.join(paths.raw_amass, "TotalCapture", subject, f"{activity}_poses.npz"))
            pose = torch.from_numpy(pose_npz['poses']).float().view(-1, 52, 3)
        except:
            failed_to_process.append(f"{subject}_{activity}")
            print(f"Failed to Process: {file}")
            continue

        pose = tc_poses[pose.shape[0]]
    
        # acc/ori and gt pose do not match in the dataset
        if acc.shape[0] < pose.shape[0]:
            pose = pose[:acc.shape[0]]
        elif acc.shape[0] > pose.shape[0]:
            acc = acc[:pose.shape[0]]
            ori = ori[:pose.shape[0]]

        # convert axis-angle to rotation matrix
        pose = math.axis_angle_to_rotation_matrix(pose).view(-1, 24, 3, 3)

        assert acc.shape[0] == ori.shape[0] and ori.shape[0] == pose.shape[0]
        accs.append(acc)    # N, 6, 3
        oris.append(ori)    # N, 6, 3, 3
        poses.append(pose)  # N, 24, 3, 3

        processed.append(file)
    
    for subject_name in subjects:
        for motion_name in sorted(os.listdir(os.path.join(paths.raw_totalcapture_official, subject_name))):
            if (subject_name == 'S5' and motion_name == 'acting3') or motion_name.startswith(".") or (f"{subject_name.lower()}_{motion_name}" in failed_to_process):
                continue   # no SMPL poses

            f = open(os.path.join(paths.raw_totalcapture_official, subject_name, motion_name, pos_file))
            line = f.readline().split('\t')
            index = torch.tensor([line.index(_) for _ in ['LeftFoot', 'RightFoot', 'Spine']])
            pos = []
            while line:
                line = f.readline()
                pos.append(torch.tensor([[float(_) for _ in p.split(' ')] for p in line.split('\t')[:-1]]))
            pos = torch.stack(pos[:-1])[:, index] * inches_to_meters
            pos[:, :, 0].neg_()
            pos[:, :, 2].neg_()
            trans.append(pos[:, 2] - pos[:1, 2])   # N, 3

    # match trans with poses
    for i in range(len(accs)):
        if accs[i].shape[0] < trans[i].shape[0]:
            trans[i] = trans[i][:accs[i].shape[0]]
        assert trans[i].shape[0] == accs[i].shape[0]

    # remove acceleration bias
    for iacc, pose, tran in zip(accs, poses, trans):
        pose = pose.view(-1, 24, 3, 3)
        _, _, vert = body_model.forward_kinematics(pose, tran=tran, calc_mesh=True)
        vacc = _syn_acc(vert[:, vi_mask])
        for imu_id in range(6):
            for i in range(3):
                d = -iacc[:, imu_id, i].mean() + vacc[:, imu_id, i].mean()
                iacc[:, imu_id, i] += d

    data = {
        'acc': accs,
        'ori': oris,
        'pose': poses,
        'tran': trans
    }
    data_path = paths.eval_dir / "totalcapture.pt"
    torch.save(data, data_path)
    print("Preprocessed TotalCapture dataset is saved at:", paths.processed_totalcapture)


def process_dipimu(split="test"):
    """Preprocess DIP for finetuning and evaluation."""
    imu_mask = [7, 8, 9, 10, 0, 2]

    test_split = ['s_09', 's_10']
    train_split = ['s_01', 's_02', 's_03', 's_04', 's_05', 's_06', 's_07', 's_08']
    subjects = train_split if split == "train" else test_split
     
    # left wrist, right wrist, left thigh, right thigh, head, pelvis
    vi_mask = torch.tensor([1961, 5424, 876, 4362, 411, 3021])
    ji_mask = torch.tensor([18, 19, 1, 2, 15, 0])

    # enable downsampling
    step = max(1, round(60 / TARGET_FPS))

    accs, oris, poses, trans, shapes, joints = [], [], [], [], [], []
    for subject_name in subjects:
        for motion_name in os.listdir(os.path.join(paths.raw_dip, subject_name)):
            try:
                path = os.path.join(paths.raw_dip, subject_name, motion_name)
                print(f"Processing: {subject_name}/{motion_name}")
                data = pickle.load(open(path, 'rb'), encoding='latin1')
                acc = torch.from_numpy(data['imu_acc'][:, imu_mask]).float()
                ori = torch.from_numpy(data['imu_ori'][:, imu_mask]).float()
                pose = torch.from_numpy(data['gt']).float()

                # fill nan with nearest neighbors
                for _ in range(4):
                    acc[1:].masked_scatter_(torch.isnan(acc[1:]), acc[:-1][torch.isnan(acc[1:])])
                    ori[1:].masked_scatter_(torch.isnan(ori[1:]), ori[:-1][torch.isnan(ori[1:])])
                    acc[:-1].masked_scatter_(torch.isnan(acc[:-1]), acc[1:][torch.isnan(acc[:-1])])
                    ori[:-1].masked_scatter_(torch.isnan(ori[:-1]), ori[1:][torch.isnan(ori[:-1])])

                # enable downsampling
                acc = acc[6:-6:step].contiguous()
                ori = ori[6:-6:step].contiguous()
                pose = pose[6:-6:step].contiguous()

                shape = torch.ones((10))
                tran = torch.zeros(pose.shape[0], 3) # dip-imu does not contain translations
                if torch.isnan(acc).sum() == 0 and torch.isnan(ori).sum() == 0 and torch.isnan(pose).sum() == 0:
                    accs.append(acc.clone())
                    oris.append(ori.clone())
                    trans.append(tran.clone())  
                    shapes.append(shape.clone()) # default shape
                    
                    # forward kinematics to get the joint position
                    p = math.axis_angle_to_rotation_matrix(pose).reshape(-1, 24, 3, 3)
                    grot, joint, vert = body_model.forward_kinematics(p, shape, tran, calc_mesh=True)
                    poses.append(p.clone())
                    joints.append(joint)
                else:
                    print(f"DIP-IMU: {subject_name}/{motion_name} has too much nan! Discard!")
            except Exception as e:
                print(f"Error processing the file: {path}.", e)


    print("Saving...")
    data = {
        'joint': joints,
        'pose': poses,
        'shape': shapes,
        'tran': trans,
        'acc': accs,
        'ori': oris,
    }
    data_path = paths.eval_dir / f"dip_{split}.pt"
    torch.save(data, data_path)
    print(f"Preprocessed DIP-IMU dataset is saved at: {data_path}")


def process_imuposer(split: str="train"):
    """Preprocess the IMUPoser dataset"""

    train_split = ['P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8']
    test_split = ['P9', 'P10']
    subjects = train_split if split == "train" else test_split

    accs, oris, poses, trans = [], [], [], []
    for pid_path in sorted(paths.raw_imuposer.iterdir()):
        if pid_path.name not in subjects:
            continue

        print(f"Processing: {pid_path.name}")
        for fpath in sorted(pid_path.iterdir()):
            with open(fpath, "rb") as f: 
                fdata = pickle.load(f)
                
                acc = fdata['imu'][:, :5*3].view(-1, 5, 3)
                ori = fdata['imu'][:, 5*3:].view(-1, 5, 3, 3)
                pose = math.axis_angle_to_rotation_matrix(fdata['pose']).view(-1, 24, 3, 3)
                tran = fdata['trans'].to(torch.float32)
                
                 # align IMUPoser global fame with DIP
                rot = torch.tensor([[[-1, 0, 0], [0, 0, 1], [0, 1, 0.]]])
                pose[:, 0] = rot.matmul(pose[:, 0])
                tran = tran.matmul(rot.squeeze())

                # ensure sizes are consistent
                assert tran.shape[0] == pose.shape[0]

                accs.append(acc)    # N, 5, 3
                oris.append(ori)    # N, 5, 3, 3
                poses.append(pose)  # N, 24, 3, 3
                trans.append(tran)  # N, 3

    print(f"# Data Processed: {len(accs)}")
    data = {
        'acc': accs,
        'ori': oris,
        'pose': poses,
        'tran': trans
    }
    data_path = paths.eval_dir / f"imuposer_{split}.pt"
    torch.save(data, data_path)


def create_directories():
    paths.processed_datasets.mkdir(exist_ok=True, parents=True)
    paths.eval_dir.mkdir(exist_ok=True, parents=True)



In [ ]:
create_directories()

In [ ]:
process_amass()

In [4]:
process_dipimu(split="train")


Processing: s_01/01.pkl
Processing: s_01/02.pkl
DIP-IMU: s_01/02.pkl has too much nan! Discard!
Processing: s_01/03.pkl
Processing: s_01/04.pkl
Processing: s_01/05.pkl
Processing: s_02/01.pkl
Processing: s_02/02.pkl
Processing: s_02/03.pkl
Processing: s_02/04.pkl
Processing: s_02/05.pkl
Processing: s_03/01.pkl
Processing: s_03/03.pkl
Processing: s_03/04.pkl
Processing: s_03/05.pkl
Processing: s_04/01.pkl
Processing: s_04/02.pkl
Processing: s_04/03.pkl
Processing: s_04/04.pkl
Processing: s_04/05.pkl
Processing: s_05/01_a.pkl
Processing: s_05/01_b.pkl
Processing: s_05/02.pkl
Processing: s_05/03.pkl
Processing: s_05/04.pkl
Processing: s_05/05.pkl
Processing: s_06/01_a.pkl
Processing: s_06/01_b.pkl
Processing: s_06/02.pkl
Processing: s_06/03.pkl
Processing: s_06/04.pkl
Processing: s_06/05.pkl
Processing: s_07/01.pkl
Processing: s_07/02.pkl
Processing: s_07/03.pkl
Processing: s_07/04.pkl
Processing: s_07/05.pkl
Processing: s_08/01_a.pkl
Processing: s_08/01_b.pkl
Processing: s_08/02.pkl
Proc

In [5]:
process_dipimu(split="test")

Processing: s_09/01_a.pkl
Processing: s_09/01_b.pkl
Processing: s_09/01_c.pkl
Processing: s_09/02_a.pkl
Processing: s_09/02_b.pkl
Processing: s_09/03_a.pkl
Processing: s_09/03_b.pkl
Processing: s_09/04.pkl
Processing: s_09/05.pkl
Processing: s_10/01_a.pkl
Processing: s_10/01_b.pkl
Processing: s_10/01_c.pkl
Processing: s_10/02.pkl
Processing: s_10/03_a.pkl
Processing: s_10/03_b.pkl
Processing: s_10/04_a.pkl
Processing: s_10/04_b.pkl
Processing: s_10/04_c.pkl
Processing: s_10/05.pkl
Saving...
Preprocessed DIP-IMU dataset is saved at: c:\deepika\2.mobileposer\MobilePoser\mobileposer\data\processed_datasets\eval\dip_test.pt


In [6]:
import torch, os
from config import paths

p = os.path.join(paths.processed_datasets, "eval", "dip_test.pt")
d = torch.load(p, map_location="cpu")
print("lens:", len(d["acc"]), len(d["ori"]), len(d["pose"]), len(d["tran"]))


lens: 19 19 19 19


C:\Users\degu03\AppData\Local\Temp\ipykernel_19284\3778834385.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  d = torch.load(p, map_location="cpu")


In [7]:
import os
import math
import numpy as np
import torch
torch.set_printoptions(sci_mode=False)
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import lightning as L
from lightning.pytorch.loggers import WandbLogger
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch import seed_everything
from argparse import ArgumentParser
from pathlib import Path
from typing import List
from tqdm import tqdm 
import wandb

from constants import MODULES
from data import PoseDataModule
from utils.file_utils import (
    get_datestring, 
    make_dir, 
    get_dir_number, 
    get_best_checkpoint
)
from config import paths, train_hypers, finetune_hypers




In [8]:

# set precision for Tensor cores
torch.set_float32_matmul_precision('medium')


class TrainingManager:
    """Manage training of MobilePoser modules."""
    def __init__(self, finetune: str=None, fast_dev_run: bool=False):
        self.finetune = finetune
        self.fast_dev_run = fast_dev_run
        self.hypers = finetune_hypers if finetune else train_hypers

    def _setup_wandb_logger(self, save_path: Path):
        wandb_logger = WandbLogger(
            project=save_path.name, 
            name=get_datestring(),
            save_dir=save_path
        ) 
        return wandb_logger

    def _setup_callbacks(self, save_path):
        checkpoint_callback = ModelCheckpoint(
                monitor="validation_step_loss",
                save_top_k=3,
                mode="min",
                verbose=False,
                dirpath=save_path, 
                save_weights_only=True,
                filename="{epoch}-{validation_step_loss:.4f}"
                )
        return checkpoint_callback

    def _setup_trainer(self, module_path: Path):
        print("Module Path: ", module_path.name, module_path)
        logger = self._setup_wandb_logger(module_path) 
        checkpoint_callback = self._setup_callbacks(module_path)
        trainer = L.Trainer(
                fast_dev_run=self.fast_dev_run,
                min_epochs=self.hypers.num_epochs,
                max_epochs=self.hypers.num_epochs,
                devices=[self.hypers.device], 
                accelerator=self.hypers.accelerator,
                logger=logger,
                callbacks=[checkpoint_callback],
                deterministic=True
                )
        return trainer

    def train_module(self, model: L.LightningModule, module_name: str, checkpoint_path: Path):
        # set the appropriate hyperparameters
        model.hypers = self.hypers 

        # create directory for module
        module_path = checkpoint_path / module_name
        make_dir(module_path)
        datamodule = PoseDataModule(finetune=self.finetune)
        trainer = self._setup_trainer(module_path)

        print()
        print("-" * 50)
        print(f"Training Module: {module_name}")
        print("-" * 50)
        print()

        try:
            trainer.fit(model, datamodule=datamodule)
        finally:
            wandb.finish()
            del model
            torch.cuda.empty_cache()


# def get_checkpoint_path(finetune: str, init_from: str):
#     if finetune:
#         # finetune from a checkpoint
#         parts = init_from.split(os.path.sep)
#         checkpoint_path = Path(os.path.join(parts[0], parts[1]))
#         finetune_dir = f"finetuned_{finetune}"
#         checkpoint_path = checkpoint_path / finetune_dir
#     else:
#         # make directory for trained models
#         dir_name = get_dir_number(paths.checkpoint) 
#         checkpoint_path = paths.checkpoint / str(dir_name)
    
#     make_dir(checkpoint_path)
#     return Path(checkpoint_path)
def get_checkpoint_path(finetune: str, init_from: str):
    if finetune:
        # finetune from a checkpoint
        parts = init_from.split(os.path.sep)
        checkpoint_path = Path(parts[0]) / parts[1]
        finetune_dir = f"finetuned_{finetune}"
        checkpoint_path = checkpoint_path / finetune_dir
    else:
        # make directory for trained models
        dir_name = get_dir_number(paths.checkpoint)
        checkpoint_path = paths.checkpoint / str(dir_name)

    # create directory if it does not exist
    checkpoint_path.mkdir(parents=True, exist_ok=True)

    return checkpoint_path


In [ ]:
seed_everything(42, workers=True)

paths.checkpoint.mkdir(exist_ok=True)
finetune = None
init_from =  "scratch"
fast_dev_run = False

module = 'poser'
checkpoint_path = get_checkpoint_path(finetune, init_from)
training_manager = TrainingManager(
    finetune=finetune,
    fast_dev_run=fast_dev_run
)

In [ ]:
seed_everything(42, workers=True)

paths.checkpoint.mkdir(exist_ok=True)
finetune = None
init_from =  "scratch"
fast_dev_run = False

module = 'joints'
checkpoint_path = get_checkpoint_path(finetune, init_from)
training_manager = TrainingManager(
    finetune=finetune,
    fast_dev_run=fast_dev_run
)

In [ ]:
MODULES

In [ ]:
if module:
    if module not in MODULES:
        raise ValueError(f"Module {module} not found.")

    module_name = module              # "poser"
    module_cls = MODULES[module_name] # models.poser.Poser (a class)

    model_dir = Path(init_from)
    model = module_cls()              # init model from scratch

    if finetune:
        model_path = get_best_checkpoint(model_dir)
        model = module_cls.from_pretrained(
            model_path=os.path.join(model_dir, model_path)
        )

    training_manager.train_module(model, module_name, checkpoint_path)
else:
    # train all modules
    for module_name, module_cls in MODULES.items():
        training_manager.train_module(module_cls(), module_name, checkpoint_path)


In [ ]:
seed_everything(42, workers=True)

paths.checkpoint.mkdir(exist_ok=True)
finetune = None
init_from =  "scratch"
fast_dev_run = False

module = 'poser'
checkpoint_path = get_checkpoint_path(finetune, init_from)
training_manager = TrainingManager(
    finetune=finetune,
    fast_dev_run=fast_dev_run
)

In [ ]:
if module:
    if module not in MODULES:
        raise ValueError(f"Module {module} not found.")

    module_name = module              # "poser"
    module_cls = MODULES[module_name] # models.poser.Poser (a class)

    model_dir = Path(init_from)
    model = module_cls()              # init model from scratch

    if finetune:
        model_path = get_best_checkpoint(model_dir)
        model = module_cls.from_pretrained(
            model_path=os.path.join(model_dir, model_path)
        )

    training_manager.train_module(model, module_name, checkpoint_path)
else:
    # train all modules
    for module_name, module_cls in MODULES.items():
        training_manager.train_module(module_cls(), module_name, checkpoint_path)


In [ ]:


if __name__ == "__main__":
    parser = ArgumentParser()
    parser.add_argument("--module", default=None)
    parser.add_argument("--fast-dev-run", action="store_true")
    parser.add_argument("--finetune", type=str, default=None)
    parser.add_argument("--init-from", nargs="?", default="scratch", type=str)
    args = parser.parse_args()

    # set seed for reproducible results
    
    # create checkpoint directory, if missing
    

    # initialize training manager
    

    # train single module
    if module:
        if module not in MODULES.keys():
            raise ValueError(f"Module {module} not found.")

        model_dir = Path(init_from)
        module = MODULES[module]
        model = module() # init model from scratch

        if finetune: 
            model_path = get_best_checkpoint(model_dir)
            model = module.from_pretrained(model_path=os.path.join(model_dir, model_path)) # load pre-trained model

        training_manager.train_module(model, module, checkpoint_path)
    else:
        # train all modules
        for module_name, module in MODULES.items():
            training_manager.train_module(module(), module_name, checkpoint_path)

In [ ]:
# python train.py --module joints --init-from checkpoints/$2/joints --finetune dip
#     python train.py --module poser --init-from checkpoints/$2/poser --finetune dip

In [5]:
seed_everything(42, workers=True)

paths.checkpoint.mkdir(exist_ok=True)
finetune = "dip"
init_from =  r"C:\deepika\2.mobileposer\MobilePoser\mobileposer\checkpoints\5\poser"
fast_dev_run = False

module = 'poser'
checkpoint_path = get_checkpoint_path(finetune, init_from)
training_manager = TrainingManager(
    finetune=finetune,
    fast_dev_run=fast_dev_run
)

if module:
    if module not in MODULES:
        raise ValueError(f"Module {module} not found.")

    module_name = module              # "poser"
    module_cls = MODULES[module_name] # models.poser.Poser (a class)

    model_dir = Path(init_from)
    model = module_cls()              # init model from scratch

    if finetune:
        print('finetuning sista')
        model_path = get_best_checkpoint(model_dir)
        model = module_cls.from_pretrained(
            model_path=os.path.join(model_dir, model_path)
        )

    training_manager.train_module(model, module_name, checkpoint_path)
else:
    # train all modules
    for module_name, module_cls in MODULES.items():
        training_manager.train_module(module_cls(), module_name, checkpoint_path)


Seed set to 42
c:\Users\degu03\AppData\Local\miniconda3\envs\mobileposer\lib\site-packages\lightning\fabric\utilities\cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
GPU availa

finetuning sista
Module Path:  poser C:deepika\finetuned_dip\poser


TPU available: False, using: 0 TPU cores
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:


--------------------------------------------------
Training Module: poser
--------------------------------------------------



wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: WARNING `resume` will be ignored since W&B syncing is set to `offline`. Starting a new run with run id 2b03vbtv.


  0%|          | 0/1 [00:00<?, ?it/s]c:\deepika\2.mobileposer\MobilePoser\mobileposer\data.py:51: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  file_data = torch.load(data_f

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

ValueError: num_samples should be a positive integer value, but got num_samples=0